<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter7/Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summarization (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
!pip install "datasets<4.0.0"
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


You will need to setup git, adapt your email and name in the following cell.

In [2]:
!git config --global user.email "alexandrupp55@gmail.com"
!git config --global user.name "Pop123-ux"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [4]:
from huggingface_hub import notebook_login
notebook_login()

## Preparing a multilingual corpus ##

We'll use the Multilingual Amazon Reviews Corpus to create our bilingual summarizer. This corpus consists of Amazon product reviews in six languages and it typically used to benchmark multilingual classifiers.

In [23]:
from datasets import load_dataset

english_dataset = load_dataset("buruzaemon/amazon_reviews_multi", "en")

spanish_dataset = load_dataset("buruzaemon/amazon_reviews_multi", "es")

print(english_dataset["train"].column_names)


es/train.jsonl.gz:   0%|          | 0.00/19.0M [00:00<?, ?B/s]

es/validation.jsonl.gz:   0%|          | 0.00/472k [00:00<?, ?B/s]

es/test.jsonl.gz:   0%|          | 0.00/476k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category']


As you can see, for each language there are 200,000 reviews for the train split, and 5,000 reviews for each of the validation and test splits. The review information we are interested in is contained in the review_body and review_title columns.

In [32]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> Title: {example['review_title']}'")
        print(f"'>> Review: {example['review_body']}'")


show_samples(english_dataset)


'>> Title: Worked in front position, not rear'
'>> Review: 3 stars because these are not rear brakes as stated in the item description. At least the mount adapter only worked on the front fork of the bike that I got it for.'

'>> Title: meh'
'>> Review: Does it’s job and it’s gorgeous but mine is falling apart, I had to basically put it together again with hot glue'

'>> Title: Can't beat these for the money'
'>> Review: Bought this for handling miscellaneous aircraft parts and hanger "stuff" that I needed to organize; it really fit the bill. The unit arrived quickly, was well packaged and arrived intact (always a good sign). There are five wall mounts-- three on the top and two on the bottom. I wanted to mount it on the wall, so all I had to do was to remove the top two layers of plastic drawers, as well as the bottom corner drawers, place it when I wanted and mark it; I then used some of the new plastic screw in wall anchors (the 50 pound variety) and it easily mounted to the wall. 

In [33]:
english_dataset.set_format('pandas')
english_df = english_dataset['train'][:]
# Show counts for top 20 products
english_df['product_category'].value_counts()[:20]

,count
product_category,
home,17679
apparel,15951
wireless,15717
other,13418
beauty,12091
drugstore,11730
kitchen,10382
toy,8745
sports,8277


In [34]:
def filter_tech(ex):
  # Ensure product_category is a string for comparison
  product_cat = str(ex['product_category']) if ex['product_category'] is not None else ""
  return (
      product_cat == 'wireless'
      or product_cat == 'electronic'
      or product_cat == 'computer'
      or product_cat == 'software'
  )

In [35]:
english_dataset.reset_format()

In [36]:
spanish_tech = spanish_dataset.filter(filter_tech)
english_tech = english_dataset.filter(filter_tech)
show_samples(english_tech)

Filter:   0%|          | 0/200000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]


'>> Title: Waste of money'
'>> Review: This watch.. I mean the phone works well on it. You can hear no problem. But the gps is like 2 miles off! Plus it’s on military time'

'>> Title: Bubbles appeared next day'
'>> Review: Was easy to put on, fit great, with no bubbles. By the next day, multiple bubbles have appeared. Unable to get them out with squeegee or fingers, as described. Wouldn't buy again.'

'>> Title: Came unscrewed, and now I have lost fitbit and the part of the necklace that held it.'
'>> Review: Came unscrewed, and now I have lost fitbit and the part of the necklace that held it. It came unscrewed as I wore it around the house and I thought it was a fluke. Wore it to work a few days later and now fitbit is gone for good.'


Hugging Face Datasets provides a handy concatenate_datasets() function that (as the name suggests) will stack two Dataset objects on top of each other. So, to create our bilingual dataset, we’ll loop over each split, concatenate the datasets for that split, and shuffle the result to ensure our model doesn’t overfit to a single language:

In [48]:
from datasets import concatenate_datasets, DatasetDict

tech_dataset = DatasetDict()

for split in english_tech.keys():
  tech_dataset[split] = concatenate_datasets(
      [english_tech[split], spanish_tech[split]]
  ).shuffle(seed=41)
  tech_dataset[split] = tech_dataset[split]

show_samples(tech_dataset)



'>> Title: No están mal'
'>> Review: Apenas si los uso.. no me resultan cómodos..se me caen..'

'>> Title: esta muy bien'
'>> Review: recomendado, buen servicio y de calidad, rapido y todo muy bien. llego en el tiempo estimado. es pequeñito y muy util'

'>> Title: Problems'
'>> Review: First thing to go wrong was that the swivel broke so it would just spin around and around. Then the clip broke and we have only had it for 5 months. NOT DURABLE for a mechanic!'


This certainly looks like a mix of English and Spanish reviews! Now that we have a training corpus, one final thing to check is the distribution of words in the reviews and their titles. This is especially important for summarization tasks, where short reference summaries in the data can bias the model to only output one or two words in the generated summaries.

To deal with this, we’ll filter out the examples with very short titles so that our model can produce more interesting summaries. Since we’re dealing with English and Spanish texts, we can use a rough heuristic to split the titles on whitespace and then use our trusty Dataset.filter() method as follows:

In [49]:
tech_dataset = tech_dataset.filter(lambda x: x["review_title"] is not None and len(x["review_title"].split()) > 2)

## Preprocessing the data ##

Our next task is to tokenize and encode our reviews and their titles. As usual, we begin by loading the tokenizer associated with the pretrained model checkpoint. We’ll use mt5-small as our checkpoint so we can fine-tune the model in a reasonable amount of time:

In [50]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [51]:
inputs = tokenizer("I loved this product!")
inputs

{'input_ids': [336, 259, 28387, 714, 5689, 309, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [52]:
tokenizer.convert_ids_to_tokens(inputs.input_ids)

['▁I', '▁', 'loved', '▁this', '▁product', '!', '</s>']

The special Unicode character ▁ and end-of-sequence token </s> indicate that we’re dealing with the SentencePiece tokenizer, which is based on the Unigram segmentation algorithm. Unigram is especially useful for multilingual corpora since it allows SentencePiece to be agnostic about accents, punctuation, and the fact that many languages, like Japanese, do not have whitespace characters.

In [54]:
max_input_length=512
max_target_length=30

def preprocess_function(examples):
  model_inputs = tokenizer(
      examples['review_body'],
      max_length=max_target_length,
      truncation=True,
  )
  labels = tokenizer(
      examples['review_title'], max_length=max_target_length, truncation=True
  )
  model_inputs['labels'] = labels['input_ids']
  return model_inputs

In [59]:
tokenized_datasets = tech_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/24206 [00:00<?, ? examples/s]

Map:   0%|          | 0/598 [00:00<?, ? examples/s]

Map:   0%|          | 0/592 [00:00<?, ? examples/s]

Now that the corpus has been preprocessed, let’s take a look at some metrics that are commonly used for summarization. As we’ll see, there is no silver bullet when it comes to measuring the quality of machine-generated text.

For summarization, one of the most commonly used metrics is the ROUGE score (short for Recall-Oriented Understudy for Gisting Evaluation). The basic idea behind this metric is to compare a generated summary against a set of reference summaries that are typically created by humans. To make this more precise, suppose we want to compare the following two summaries:

In [60]:
generated_summary = 'I love tech products'
reference_summary = 'I love to use tech products'

One way to compare them could be to count the number of overlapping words, which in this case would be 6. However, this is a bit crude, so instead ROUGE is based on computing the precision and recall scores for the overlap.

In [61]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a483e82fcec3563f9897bf9660535d81e84621f7c838a9048f33e4570743fd87
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [63]:
import evaluate

rouge_score = evaluate.load('rouge')

Then we can use the rouge_score.compute() function to calculate all the metrics at once:

In [65]:
scores = rouge_score.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

{'rouge1': np.float64(0.8),
 'rouge2': np.float64(0.5),
 'rougeL': np.float64(0.8),
 'rougeLsum': np.float64(0.8)}

The rouge1 variant is the overlap of unigrams — this is just a fancy way of saying the overlap of words and is exactly the metric we’ve discussed above; rouge2 measures the overlap between bigrams (think the overlap of pairs of words), while rougeL and rougeLsum measure the longest matching sequences of words by looking for the longest common substrings in the generated and reference summaries.

We’ll use these ROUGE scores to track the performance of our model, but before doing that let’s do something every good NLP practitioner should do: create a strong, yet simple baseline!

## Creating a strong baseline ##

A common baseline used for text summarization is to simply take the first three sentences of an article, often called the lead-3 baseline.
We could use full stops to track the sentence boundaries, but this will fail on acronyms - so instead we'll use the nltk library, which includes a better algorithm to handle these cases

In [68]:
!pip install nltk

In [72]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

Next, we import the sentence tokenizer from nltk and create a simple function to extract the first three sentences in a review. The convention in text summarization is to separate each summary with a newline, so let’s also include this and test it on a training example:

In [74]:
from nltk.tokenize import sent_tokenize

def three_sentence_summary(text):
  return "\n".join(sent_tokenize(text)[:3])

print(three_sentence_summary(tech_dataset['train'][1]['review_body']))

Baratisimo, y muy util.
En mi caso lo queria para meter un un pendrive al firestick de amazon y va de escandalo.
No lo he probado para otros usos pero me imagino que con moviles y tablets otg tambien tiene que funcionar.


This seems to work, so let’s now implement a function that extracts these “summaries” from a dataset and computes the ROUGE scores for the baseline:

In [75]:
def evaluate_baseline(dataset, metric):
  summaries = [three_sentence_summary(text) for text in dataset['review_body']]
  return metric.compute(predictions=summaries, references=dataset['review_title'])

We can then use this function to compute the ROUGE scores over the validation set and prettify them a bit using Pandas:

In [76]:
import pandas as pd

score = evaluate_baseline(tech_dataset['validation'], rouge_score)
rouge_names = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
rouge_dict = dict((rn, round(score[rn] * 100, 2)) for rn in rouge_names)
rouge_dict

{'rouge1': np.float64(18.56),
 'rouge2': np.float64(10.15),
 'rougeL': np.float64(17.14),
 'rougeLsum': np.float64(17.63)}

We can see that the rouge2 score is significantly lower than the rest; this likely reflects the fact that review titles are typically concise and so the lead-3 baseline is too verbose. Now that we have a good baseline to work from, let’s turn our attention toward fine-tuning mT5!

## Fine-tuning mT5 with the Trainer API ##

The first thing we need to do is load the pretrained model from the mt5-small checkpoint. Since summarization is a sequence-to-sequence task, we can load the model with the AutoModelForSeq2SeqLM class, which will automatically download and cache the weights:

In [77]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

We'll need to generate summaries in order to compute ROUGE scores during training. Fortunately, Hugging Face Transformers provides Seq2SeqTrainingArguments and Seq2SeqTrainer classes that can do this for us automatically! To see how this works, let's first define the hyperparameters and other arguments for our experiments:

In [79]:
from transformers import Seq2SeqTrainingArguments

batch_size = 8
num_train_epochs = 8
# Show the training loss with every epoch
logging_steps = len(tokenized_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-amazon-en-es",
    learning_rate=5.6e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=num_train_epochs,
    predict_with_generate=True,
    logging_steps=logging_steps,
    push_to_hub=True,
)

The next thing we need to do is provide the trainer with a compute_metrics() function so that we can evaluate our model during training. For summarization this is a bit more involved than simply calling rouge_score.compute() on the model’s predictions, since we need to decode the outputs and labels into text before we can compute the ROUGE scores. The following function does exactly that, and also makes use of the sent_tokenize() function from nltk to separate the summary sentences with newlines:

In [82]:
import numpy as np

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  # Decode generated summaries into text
  decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
  # Replace -100 in the labels as we can't decode them
  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
  # Decode reference summaries into text
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
  # ROUGE expects a newline after each sentence
  decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in decoded_preds]
  decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in decoded_labels]
  # Compute ROUGE scores
  result = rouge_score.compute(
      predictions=decoded_preds, references=decoded_labels, use_stemmer=True
  )
  # Extract the median scores
  result = {key: value.mid.fmeasure * 100 for key, value in result.items()} # The median of the F1 Score
  return {k: round(v, 4) for k, v in result.items()}

Next, we need to define a data collator for our sequence-to-sequence task. Since mT5 is an encoder-decoder Transformer model, one subtlety with preparing our batches is that during decoding we need to shift the labels to the right by one.

This is similar to how masked self-attention is applied to the inputs in a task like causal language modeling.

In [83]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Let’s see what this collator produces when fed a small batch of examples. First, we need to remove the columns with strings because the collator won’t know how to pad these elements:

In [84]:
tokenized_datasets = tokenized_datasets.remove_columns(
    tech_dataset['train'].column_names
)

Since the collator expects a list of dicts, where each dict represents a single example in the dataset, we also need to wrangle the data into the expected format before passing it to the data collator:

In [85]:
features = [tokenized_datasets['train'][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[  4295,  22358,  16831,    375,    259,  42763,   4164,    450,    362,
            288,    505,   2784,    426,   6775,    362, 185048,    519,    707,
            319,    269,  68464,    362,   6775,   2784,    375,    707,  23982,
          34114,    259,      1],
        [ 13178, 165819,    261,    259,    276,   3861,  48059,    260,    642,
            658,   2117,    707,    319,   3040,    435,  16126,    335,    335,
           3617,  54756,    440,  10112,  46038,    269,  49799,    259,    276,
            712,    269,      1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]), 'labels': tensor([[11575,   262,   362,  6775,     1,  -100,  -100],
        [13178,   268,   259,   276, 48059,   260,     1]]), 'decoder_input_ids': tensor([[    0, 11575,   262,   362,  677

The main thing to notice here is that the first example is longer than the second one, so the input_ids and attention_mask of the second example have been padded on the right with a [PAD] token (whose ID is 0). Similarly, we can see that the labels have been padded with -100s, to make sure the padding tokens are ignored by the loss function. And finally, we can see a new decoder_input_ids which has shifted the labels to the right by inserting a [PAD] token in the first entry.

We finally have all the ingredients we need to train with! We now simply need to instantiate the trainer with the standard arguments:

In [87]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.evaluate()

In [ ]:
trainer.push_to_hub(commit_message="Training complete", tags='summarization')

# Fine-tuning mT5 with Hugging Face Accelerate #

## Preparing everything for training ##

The first thing we need to do is create a DataLoader for each of our splits. Since the PyTorch dataloaders expect batches of tensors, we need to set this format to "torch" in our datasets

In [ ]:
tokenized_datasets.set_format("torch")

Now that we’ve got datasets consisting of just tensors, the next thing to do is instantiate the DataCollatorForSeq2Seq again. For this we need to provide a fresh version of the model, so let’s load it again from our cache:

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

We can then instantiate the data collator and use this to define our dataloaders:

In [ ]:
from torch.utils.data import DataLoader

batch_size=8
train_dataloader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=batch_size,
)
eval_dataloader=DataLoader(
    tokenized_datasets['validation'],
    collate_fn=data_collator,
    batch_size=batch_size,
)

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=2e-5)

Finally, we feed our model, optimizer, and dataloaders to the accelerator.prepare() method:

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()

model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

Now that we’ve prepared our objects, there are three remaining things to do:

- Define the learning rate schedule.
- Implement a function to post-process the summaries for evaluation.
- Create a repository on the Hub that we can push our model to.

For the learning rate schedule, we’ll use the standard linear one from previous sections:

In [ ]:
from transformers import get_scheduler

epochs=10
update_steps = len(train_dataloader)
train_steps = epochs * update_steps
lr_scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

For post-processing, we need a function that splits the generated summaries into sentences that are separated by newlines. This is the format the ROUGE metric expects, and we can achieve this with the following:

In [ ]:
def postprocess_text(preds, labels):
  preds = [pred.strip() for pred in preds]
  labels = [label.strip() for label in labels]

  # ROUGE expects a newline after each sentence
  preds = ["\n".join(ntlk.sent_tokenize(pre)) for pred in preds]
  labels = ["\n".join(ntlk.sent_tokenize(label)) for label in labels]

  return preds, labels

Now we can use this repository name to clone a local version to our results directory that will store the training artifacts:

In [ ]:
from huggingface_hub import get_full_repo_name

model_name = "test-bert-finetuned-squad-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

This will allow us to push the artifacts back to the Hub by calling the repo.push_to_hub() method during training! Let’s now wrap up our analysis by writing out the training loop.

## Training loop ##

In [ ]:
from tqdm.auto import tqdm
import torch
import numpy as np

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            generated_tokens = accelerator.pad_across_processes(
                generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
            )
            labels = batch["labels"]

            # If we did not pad to max length, we need to pad the labels too
            labels = accelerator.pad_across_processes(
                batch["labels"], dim=1, pad_index=tokenizer.pad_token_id
            )

            generated_tokens = accelerator.gather(generated_tokens).cpu().numpy()
            labels = accelerator.gather(labels).cpu().numpy()

            # Replace -100 in the labels as we can't decode them
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
            if isinstance(generated_tokens, tuple):
                generated_tokens = generated_tokens[0]
            decoded_preds = tokenizer.batch_decode(
                generated_tokens, skip_special_tokens=True
            )
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            decoded_preds, decoded_labels = postprocess_text(
                decoded_preds, decoded_labels
            )

            rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    # Compute metrics
    result = rouge_score.compute()
    # Extract the median ROUGE scores
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    result = {k: round(v, 4) for k, v in result.items()}
    print(f"Epoch {epoch}:", result)

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
      tokenizer.save_pretrained(output_dir)
      repo.push_to_hub(
          commit_message=f"Training in progress epoch {epoch}",
          blocking=False,
      )

## Using the fine-tuned model ##

In [ ]:
from transformers import pipeline

hub_model_id = "pop123/mt5-small-finetuned-amazon-en-es"
summerizer = pipeline('summarization', model=hub_model_id)

We can feed some examples from the test set (which the model has not seen) to our pipeline to get a feel for the quality of the summaries. First let’s implement a simple function to show the review, title, and generated summary together:

In [ ]:
def print_summary(idx):
  review = tech_dataset['test'][idx]['review_body']
  title = tech_dataset['test'][idx]['review_title']
  summary = summarizer(tech_dataset['test'][idx]['review_body'])[0]['summary_text']
  print(f"'>>> Review: {review}'")
  print(f"\n'>>> Title: {title}'")
  print(f"\n'>>> Summary: {summary}'")

Let’s take a look at one of the English examples we get:

In [ ]:
print_summary(100)

This is not too bad! We can see that our model has actually been able to perform abstractive summarization by augmenting parts of the review with new words. And perhaps the coolest aspect of our model is that it is bilingual, so we can also generate summaries of Spanish reviews:

In [ ]:
print_summary(0)